# Task 1: Basics - loading data, visualization, segmentation, filtering

## 1.1 Loading data and filtering it

The processing order for optical motion capture data is usually as follows:
1. Record OMC data (marker trajectories) + force plate data (ground reaction forces)
2. Label markers and fill gaps in the marker trajectories (already done for you)
3. Scale musculoskeletal model to the subject (more info [here](https://opensimconfluence.atlassian.net/wiki/spaces/OpenSim/pages/53089741/Tutorial+3+-+Scaling+Inverse+Kinematics+and+Inverse+Dynamics))
4. Inverse kinematics: calculate joint angles from marker trajectories
5. Filter marker trajectories and joint angles -- to remove high-frequency noise. Always filter whole trajectories, not only segments ()
6. Inverse dynamics: calculate joint moments from joint angles and ground reaction forces

**To Do:**

Go to filter.py and implement a 4th zero-lag Butterworth low-pass filter with a cutoff frequency of 6 Hz. You can use the scipy functions `scipy.signal.butter` and `scipy.signal.filtfilt` for this.

Then run the next cell to visualize the effect of filtering on the marker trajectories. You should see that high-frequency noise is removed, but the overall shape of the signal is preserved.

In [ ]:
from utils import filter, load_data
import matplotlib.pyplot as plt

grf_data = load_data.load_grf_data()
# This filter needs to be implemented by you in utils/filter.py first!
filtered_grf_data = filter.butterworth_lowpass_filter(grf_data, cutoff=6, fs=100, order=4)
range = slice(1000, 1200)
plt.plot(grf_data['time'][range], grf_data['ground_force_vy'][range], label='unfiltered')
plt.plot(filtered_grf_data['time'][range], filtered_grf_data['ground_force_vy'][range], label='filtered')
plt.xlabel('time (s)')
plt.ylabel('ground force vertical (N)')
plt.legend()

Run `pytest tests.py` to check if the first part of your implementation is correct - 2 tests should pass now.

## 1.2 Segmenting gait cycles
We can use the vertical ground reaction force (GRF) to identify heel strikes and toe offs. A common threshold is 20 N: if the vertical GRF exceeds this value, the foot is on the ground. Always use the unfiltered GRF data for this.

Next cell plots the points we roughly search for:

In [ ]:
plt.plot(grf_data['time'][range], grf_data['ground_force_vy'][range], label='unfiltered')
plt.plot(10.08, 0, 'ro')  # heel strike 1
plt.plot(11.03, 0, 'ro')  # heel strike 2

**To Do:**

Now, implement the function `segment_gait_cycles` in `utils/segment.py` that segments the data into individual gait cycles based on heel strikes. Use a threshold of 60 N on the unfiltered vertical ground reaction force to identify heel strikes. The function should also clean the data: remove gait cycles that do not reach a minimum force of 300 N (to remove partial steps, noise outliers, etc.). Then, remove all gait cycles that are +- 2 standard deviations away from the mean gait cycle duration (to remove outliers). The function should return a list of data frames, each containing one gait cycle. 

Then execute the next cell to see all gait cycles overlaid. Do you see a clear pattern as in the lecture slides?

In [ ]:
from utils import segment
gait_cycles = segment.segment_gait_cycles(grf_data.ground_force_vy, data=grf_data, threshold=60)
for i, cycle in enumerate(gait_cycles):
    plt.plot(cycle['time'], cycle['ground_force_vy'])
plt.xlabel('time (s)')
plt.ylabel('ground force vertical (N)')
plt.legend()
print(f'Number of gait cycles remaining: {len(gait_cycles)}')

## 1.3 Ensemble averaging gait cycles

**To Do:**

Implement a function `ensemble_average` in `utils/segment.py` that takes a list of data frames (gait cycles) and returns the ensemble average and standard deviation. The function should first resample each gait cycle to 100 data points (using linear interpolation), then calculate the mean and standard deviation across all cycles.

Then execute the next cell to visualize the ensemble average and standard deviation of the vertical ground reaction force across all gait cycles. You should see a smooth curve representing the average GRF pattern during walking, with shaded areas indicating variability across cycles.

This should look like the plots in the lecture slides.

In [ ]:
ensemble_average, ensemble_std = segment.ensemble_average(gait_cycles)
plt.plot(ensemble_average['time'], ensemble_average['ground_force_vy'], label='ensemble average')
plt.fill_between(ensemble_average['time'], 
                 ensemble_average['ground_force_vy'] - ensemble_std['ground_force_vy'],
                 ensemble_average['ground_force_vy'] + ensemble_std['ground_force_vy'],
                 color='gray', alpha=0.5, label='±1 std dev')
plt.xlabel('time (s)')
plt.ylabel('ground force vertical (N)')
plt.legend()

If that all worked, check out the test function again: `pytest tests.py` - 1 more test should pass now!

## 1.4 To do: Visualize joint angles

**To Do:**

Visualize the following ensemble averages with standard deviations:
- Hip flexion/extension
- Knee flexion/extension
- Ankle dorsiflexion/plantarflexion
- Ground reaction force (horizontal and vertical)
Make it look nice (labels, legends, etc.) and compare to the plots in the lecture slides. Do they look similar?

In [ ]:
"""
    Plot joint angles here
"""


Tests running? Plots done? Great! You are done with assignment 1. 